# 🧠 โครงข่ายประสาทเทียมแบบคอนโวลูชัน (CNN) และการสกัดคุณลักษณะ (Feature Extraction)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Convolutional Neural Networks (CNNs)**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายข้อจำกัดของ MLP กับข้อมูลรูปภาพ และพลังของพื้นที่การรับรู้เฉพาะที่ (local receptive fields) และการแบ่งปันน้ำหนัก (weight sharing)
2. พัฒนา **2D Convolution จากศูนย์ด้วย NumPy** ที่รองรับ padding และ stride แบบกำหนดเอง
3. ใช้ **ตัวกรองตรวจจับขอบ Sobel (Sobel Edge Detection Filter)** กับภาพขอบจำลองเพื่อสังเกตการสกัดคุณลักษณะ
4. แสดงภาพของรูปภาพต้นฉบับ, ตัวกรอง Sobel (Sobel kernel) และแผนที่ลักษณะผลลัพธ์ (output feature map)
5. พัฒนาสูตรคำนวณ **ตัวคำนวณรูปร่างผลลัพธ์ของ CNN (CNN Output Shape Calculator)**
6. เชื่อมโยงการทำงานเหล่านี้เข้ากับการทำคอนโวลูชันและการสุ่มตัวอย่างลง (downsampling) ในโครงสร้างหลัก (backbone) ของ YOLO

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การพัฒนา 2D Convolution จากศูนย์

เรามาเขียนฟังก์ชัน NumPy ที่ทำการเสริมขอบ (pad) อินพุต และทำการหาผลคูณจุดแบบองค์ประกอบต่อองค์ประกอบ (element-wise dot product) ของหน้าต่างที่เลื่อนไป (sliding-window) กันครับ

In [ ]:
def convolve2d_scratch(img, kernel, stride=1, padding=0):
    h_in, w_in = img.shape
    f_h, f_w = kernel.shape
    
    if padding > 0:
        img_padded = np.pad(img, padding, mode='constant', constant_values=0)
    else:
        img_padded = img.copy()
        
    h_out = int(np.floor((h_in - f_h + 2 * padding) / stride) + 1)
    w_out = int(np.floor((w_in - f_w + 2 * padding) / stride) + 1)
    
    output = np.zeros((h_out, w_out))
    
    for i in range(h_out):
        for j in range(w_out):
            r_start = i * stride
            r_end = r_start + f_h
            c_start = j * stride
            c_end = c_start + f_w
            
            region = img_padded[r_start:r_end, c_start:c_end]
            output[i, j] = np.sum(region * kernel)
            
    return output

## 2. กรณีศึกษา: การตรวจจับขอบแนวตั้งด้วย Sobel

เรามาสร้างภาพจำลองขนาด $6 \times 6$ ที่มีการเปลี่ยนค่าในแนวตั้งอย่างฉับพลัน (จากค่า 10 ไปเป็น 0) ซึ่งแสดงถึงเส้นขอบ และใช้ตัวกรองแนวตั้ง Sobel:
$$K = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$

In [ ]:
img = np.array([
    [10, 10, 10, 0, 0, 0],
    [10, 10, 10, 0, 0, 0],
    [10, 10, 10, 0, 0, 0],
    [10, 10, 10, 0, 0, 0],
    [10, 10, 10, 0, 0, 0],
    [10, 10, 10, 0, 0, 0]
])

kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

feature_map = convolve2d_scratch(img, kernel, stride=1, padding=1)

print("Original Image:\n", img)
print("\nSobel Filter Output Feature Map:\n", feature_map)

ดูผลลัพธ์ที่ได้สิครับ! บริเวณที่เป็นพื้นที่ราบเรียบจะถูกจับคู่เป็น 0 ในขณะที่คอลัมน์ที่เป็นขอบแนวตั้งจะถูกเน้นด้วยค่าที่สูง ซึ่งช่วยให้ตรวจจับตำแหน่งของขอบเขตได้อย่างแม่นยำ!

## 3. การแสดงผลภาพการสกัดคุณลักษณะ

เรามาพล็อตกราฟทั้งสามเมทริกซ์นี้ข้างกันโดยใช้แผนภูมิความร้อน (heatmap) เพื่อแสดงการเปลี่ยนแปลงของค่าต่างๆ ให้เห็นภาพชัดเจนกันครับ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image (Edge)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(kernel, cmap='bwr', vmin=-2, vmax=2)
axes[1].set_title('Sobel Kernel (Filter)')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(feature_map, cmap='gray')
axes[2].set_title('Output Feature Map')
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_xticks(np.arange(6))
    ax.set_yticks(np.arange(6))

axes[1].set_xticks(np.arange(3))
axes[1].set_yticks(np.arange(3))

plt.tight_layout()
plt.show()

## 4. ตัวคำนวณรูปร่างผลลัพธ์ของ CNN

เรามาสร้างฟังก์ชันช่วยคำนวณขนาดมิติที่เป็นมาตรฐานกันครับ:
$$W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} - F + 2P}{S} \right\rfloor + 1$$

In [ ]:
def calculate_output_shape(w_in, kernel_size, padding, stride):
    return int(np.floor((w_in - kernel_size + 2 * padding) / stride) + 1)

print("Calculated Dimension:", calculate_output_shape(640, 3, 1, 2))

## 💡 การเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **การสุ่มตัวอย่างลงโดยไม่ใช้ Pooling:** โครงข่าย CNN รุ่นเก่า (เช่น VGG) จะใช้เลเยอร์ Max Pooling เพื่อลดขนาดทางพื้นที่ (spatial size) แต่โครงสร้าง YOLO ในปัจจุบัน (YOLOv5 ถึง YOLO11) จะทำการลดขนาดข้อมูลผ่าน **Strided Convolutions** (เช่น การใช้ `stride=2` ภายในบล็อก `Conv`) ซึ่งช่วยลดมิติทางพื้นที่ในขณะที่เรียนรู้พารามิเตอร์ไปพร้อมๆ กัน ทำให้รักษาความแม่นยำของตำแหน่งได้ดียิ่งขึ้น
*   **พื้นที่การรับรู้ (Receptive Field):** การวางซ้อนการคอนโวลูชันหลายๆ เลเยอร์จะช่วยเพิ่มพื้นที่การรับรู้ ช่วยให้เลเยอร์ที่อยู่ลึกสามารถมองเห็นพื้นที่ของภาพอินพุตที่กว้างขึ้น ซึ่งจำเป็นสำหรับการตรวจจับวัตถุขนาดใหญ่